# Segundo modelo — predicción de rango de elección en el Draft

Aquí vamos a atacar el problema más difícil: predecir el rango exacto de pick entre 7 clases muy desbalanceadas.

El primer modelo (XGBoost balanceado) tenía un macro F1 de ~0.24 — prácticamente azar. El problema principal es el desbalanceo extremo (ND = 70%, ~7% por cada rango drafteado).

Las estrategias de este segundo modelo:

1. **RandomizedSearchCV** sobre XGBoost con espacio de búsqueda amplio
2. **LightGBM** — más robusto con clases poco representadas
3. **Reducción de clases**: fusionamos R1 completa (1-30) y R2 completa (31-60) antes de afinar el rango → el modelo trabaja con 3 clases limpias primero y luego el rango
4. **Selección de variables** con top features del modelo optimizado


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier
import lightgbm as lgb


## Cargo los datos

In [ ]:
ncaa = pd.read_csv('../../datos/procesados/ncaa_final.csv')

X = ncaa.drop(columns=['ronda', 'rango_pick'])
X = pd.get_dummies(X, columns=['posicion'], drop_first=False)

le_target_eleccion = LabelEncoder()
y_enc = le_target_eleccion.fit_transform(ncaa['rango_pick'])

print("Clases:", le_target_eleccion.classes_)
print("Distribución:")
print(pd.Series(y_enc).value_counts(normalize=True).round(3).to_dict())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size=0.15,
    random_state=11,
    stratify=y_enc
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
n_clases = len(le_target_eleccion.classes_)
print(f"Número de clases: {n_clases}")


## Estrategia 1 — RandomizedSearchCV sobre XGBoost

Con 7 clases tan desbalanceadas la búsqueda aleatoria puede ayudar a encontrar combinaciones de `max_depth` y `learning_rate` que regularicen el modelo sin colapsar en ND siempre.

Usamos `f1_macro` como métrica de scoring para penalizar igual todos los rangos.

In [ ]:
pesos_train = compute_sample_weight(class_weight='balanced', y=y_train)

param_dist = {
    'n_estimators':     [100, 200, 300, 400],
    'max_depth':        [3, 4, 5, 6],
    'learning_rate':    [0.01, 0.05, 0.1, 0.15],
    'subsample':        [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma':            [0, 0.1, 0.2, 0.5],
    'reg_alpha':        [0, 0.1, 0.5, 1.0],   # regularización L1
    'reg_lambda':       [1, 1.5, 2.0]          # regularización L2
}

xgb_base = XGBClassifier(
    objective='multi:softmax',
    num_class=n_clases,
    random_state=11,
    eval_metric='mlogloss'
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=11)

rnd_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1_macro',
    cv=cv,
    random_state=11,
    n_jobs=-1,
    verbose=1
)

rnd_search.fit(X_train, y_train, sample_weight=pesos_train)

print("\nMejores hiperparámetros:")
for k, v in rnd_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nMejor F1 macro en CV: {rnd_search.best_score_:.4f}")


In [ ]:
mejor_xgb = rnd_search.best_estimator_
y_pred_rxgb = mejor_xgb.predict(X_test)

print("=== XGBoost + RandomizedSearch ===")
print(classification_report(y_test, y_pred_rxgb, target_names=le_target_eleccion.classes_))


In [ ]:
cm_rxgb = confusion_matrix(y_test, y_pred_rxgb)
disp = ConfusionMatrixDisplay(cm_rxgb, display_labels=le_target_eleccion.classes_)
fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(cmap='Blues', ax=ax)
plt.title('Matriz de Confusión — XGBoost + RandomizedSearch (rango pick)')
plt.tight_layout()
plt.show()


## Estrategia 2 — LightGBM con parámetros ajustados

LightGBM tiene la ventaja de `num_leaves` (controla la complejidad del árbol de forma más fina que `max_depth`) y `min_data_in_leaf` (regulariza los nodos hoja, muy útil cuando hay rangos con solo 14-16 muestras en test).

In [ ]:
lgbm_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=n_clases,
    n_estimators=400,
    max_depth=6,
    num_leaves=31,         # controla complejidad del árbol (< 2^max_depth para regularizar)
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=5,   # con clases pequeñas, bajo este valor para no perder hojas
    class_weight='balanced',
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=11,
    verbose=-1
)

lgbm_model.fit(X_train, y_train)
y_pred_lgbm = lgbm_model.predict(X_test)

print("=== LightGBM balanceado ===")
print(classification_report(y_test, y_pred_lgbm, target_names=le_target_eleccion.classes_))


In [ ]:
cm_lgbm = confusion_matrix(y_test, y_pred_lgbm)
disp_lgbm = ConfusionMatrixDisplay(cm_lgbm, display_labels=le_target_eleccion.classes_)
fig, ax = plt.subplots(figsize=(9, 7))
disp_lgbm.plot(cmap='Blues', ax=ax)
plt.title('Matriz de Confusión — LightGBM (rango pick)')
plt.tight_layout()
plt.show()


## Estrategia 3 — Reducción de clases (3 → 7)

El desbalanceo es tan extremo que los rangos individuales son casi imposibles de aprender. Una estrategia es trabajar con 3 clases primero (ND / R1 / R2), como en el modelo de ronda, y luego aplicar un segundo clasificador solo sobre los jugadores predichos como drafteados.

Aquí construimos ese **segundo nivel**: dado que el modelo de ronda ya clasifica bien ND, entrenamos un modelo solo sobre los ~600 jugadores drafteados para predecir si son R1 (1-30) o R2 (31-60). Es un problema binario mucho más manejable.

In [ ]:
# filtro solo los jugadores drafteados del dataset completo
ncaa_drafteados = ncaa[ncaa['ronda'] != 'ND'].copy()

print(f"Jugadores drafteados en el dataset: {len(ncaa_drafteados)}")
print("Distribución por rango:")
print(ncaa_drafteados['rango_pick'].value_counts())


In [ ]:
# preparo X e y solo para drafteados
X_draft = ncaa_drafteados.drop(columns=['ronda', 'rango_pick'])
X_draft = pd.get_dummies(X_draft, columns=['posicion'], drop_first=False)

le_rango_draft = LabelEncoder()
y_draft_enc = le_rango_draft.fit_transform(ncaa_drafteados['rango_pick'])

print("Clases rango (solo drafteados):", le_rango_draft.classes_)
print("Distribución:", pd.Series(y_draft_enc).value_counts(normalize=True).round(3).to_dict())


In [ ]:
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_draft, y_draft_enc,
    test_size=0.15,
    random_state=11,
    stratify=y_draft_enc
)

print(f"Train drafteados: {X_train_d.shape} | Test: {X_test_d.shape}")


In [ ]:
# LightGBM entrenado solo sobre drafteados — problema de 6 clases en vez de 7
# el desbalanceo desaparece porque quitamos el 70% de ND
lgbm_draft_only = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(le_rango_draft.classes_),
    n_estimators=300,
    max_depth=5,
    num_leaves=20,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=5,
    class_weight='balanced',
    random_state=11,
    verbose=-1
)

lgbm_draft_only.fit(X_train_d, y_train_d)
y_pred_do = lgbm_draft_only.predict(X_test_d)

print("=== LightGBM solo sobre jugadores drafteados ===")
print(classification_report(y_test_d, y_pred_do, target_names=le_rango_draft.classes_))


In [ ]:
cm_do = confusion_matrix(y_test_d, y_pred_do)
disp_do = ConfusionMatrixDisplay(cm_do, display_labels=le_rango_draft.classes_)
fig, ax = plt.subplots(figsize=(9, 7))
disp_do.plot(cmap='Blues', ax=ax)
plt.title('Matriz de Confusión — LightGBM solo drafteados')
plt.tight_layout()
plt.show()


## Estrategia 4 — Feature importance y top variables

Igual que en el modelo de ronda, usamos las variables más importantes del XGBoost optimizado para entrenar un LightGBM más limpio.

In [ ]:
importancias = pd.Series(
    mejor_xgb.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0a1019')
ax.set_facecolor('#0a1019')
top20 = importancias.head(20)
ax.barh(top20.index[::-1], top20.values[::-1], color='#3B6D11', alpha=0.85)
ax.set_xlabel('importancia', color='white')
ax.set_title('Top 20 variables — XGBoost optimizado (rango pick)', color='white', fontsize=13)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#ffffff22')
plt.tight_layout()
plt.show()


In [ ]:
top15_vars = importancias.head(15).index.tolist()
print("Top 15 variables seleccionadas:")
for v in top15_vars: print(f"  - {v}")

X_train_top = X_train[top15_vars]
X_test_top  = X_test[top15_vars]

lgbm_top = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=n_clases,
    n_estimators=400,
    max_depth=6,
    num_leaves=25,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=5,
    class_weight='balanced',
    random_state=11,
    verbose=-1
)

lgbm_top.fit(X_train_top, y_train)
y_pred_top = lgbm_top.predict(X_test_top)

print("\n=== LightGBM top 15 variables ===")
print(classification_report(y_test, y_pred_top, target_names=le_target_eleccion.classes_))


## Comparativa final

In [ ]:
# reconstruyo el modelo base del primer notebook para comparar
modelo_base_ref = XGBClassifier(
    objective='multi:softmax', num_class=n_clases,
    n_estimators=100, max_depth=4, learning_rate=0.1,
    random_state=42, eval_metric='mlogloss'
)
pesos_base = compute_sample_weight(class_weight='balanced', y=y_train)
modelo_base_ref.fit(X_train, y_train, sample_weight=pesos_base)
y_pred_base = modelo_base_ref.predict(X_test)

modelos_nombre = ['Base (XGB)', 'XGB + RndSearch', 'LightGBM', 'LightGBM top15']
preds_lista    = [y_pred_base, y_pred_rxgb, y_pred_lgbm, y_pred_top]

resultados = []
for nombre, preds in zip(modelos_nombre, preds_lista):
    rep = classification_report(y_test, preds, target_names=le_target_eleccion.classes_, output_dict=True)
    resultados.append({
        'modelo':    nombre,
        'macro_f1':  rep['macro avg']['f1-score'],
        'f1_ND':     rep['ND']['f1-score'],
        'f1_1-10':   rep['1-10']['f1-score'],
        'f1_11-20':  rep['11-20']['f1-score'],
        'f1_51-60':  rep['51-60']['f1-score'],
    })

df_res = pd.DataFrame(resultados)
print(df_res.to_string(index=False))


In [ ]:
# gráfico comparativo
fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#0a1019')
ax.set_facecolor('#0a1019')

x_pos = np.arange(len(modelos_nombre))
cols_m  = ['macro_f1', 'f1_1-10', 'f1_51-60']
colores_c = ['#f97316', '#A32D2D', '#854F0B']
etiq_c   = ['macro F1', 'F1 1-10', 'F1 51-60']
ancho_c  = 0.22

for i, (col, color, etiq) in enumerate(zip(cols_m, colores_c, etiq_c)):
    vals = [df_res.loc[j, col] for j in range(len(modelos_nombre))]
    barras = ax.bar(x_pos + i * ancho_c, vals, ancho_c, label=etiq, color=color, alpha=0.85)
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                f'{b.get_height():.2f}', ha='center', va='bottom', fontsize=8, color='white')

ax.set_xticks(x_pos + ancho_c)
ax.set_xticklabels(modelos_nombre, color='white', fontsize=10)
ax.set_ylim(0, 1.0)
ax.set_ylabel('valor', color='white')
ax.set_title('Comparativa de modelos — predicción de rango pick', color='white', fontsize=13, pad=12)
ax.tick_params(colors='white')
ax.legend(facecolor='#1a2535', labelcolor='white', fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor('#ffffff22')
plt.tight_layout()
plt.show()


## Guardo modelos

## Estrategia 5 — Random Forest con pesos balanceados

Mismo planteamiento que en el modelo de ronda pero con 7 clases. Random Forest suele ser más robusto que SVM o Regresión Logística ante desbalanceo extremo gracias al bagging — cada árbol ve una muestra diferente del dataset.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,    # bajo para no perder las clases minoritarias
    class_weight='balanced',
    random_state=11,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("=== Random Forest balanceado ===")
print(classification_report(y_test, y_pred_rf, target_names=le_target_eleccion.classes_))


In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(cm_rf, display_labels=le_target_eleccion.classes_)
fig, ax = plt.subplots(figsize=(9, 7))
disp_rf.plot(cmap='Blues', ax=ax)
plt.title('Matriz de Confusión — Random Forest (rango pick)')
plt.tight_layout()
plt.show()


## Estrategia 6 — SVM con kernel RBF

SVM es el modelo más costoso computacionalmente con 7 clases (usa One vs One internamente: 21 clasificadores binarios). Con datasets de ~2000 filas es manejable. El pipeline incluye StandardScaler obligatorio.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        class_weight='balanced',
        probability=True,
        random_state=11
    ))
])

svm_pipeline.fit(X_train, y_train)
y_pred_svm = svm_pipeline.predict(X_test)

print("=== SVM kernel RBF ===")
print(classification_report(y_test, y_pred_svm, target_names=le_target_eleccion.classes_))


In [ ]:
cm_svm = confusion_matrix(y_test, y_pred_svm)
disp_svm = ConfusionMatrixDisplay(cm_svm, display_labels=le_target_eleccion.classes_)
fig, ax = plt.subplots(figsize=(9, 7))
disp_svm.plot(cmap='Blues', ax=ax)
plt.title('Matriz de Confusión — SVM RBF (rango pick)')
plt.tight_layout()
plt.show()


## Estrategia 7 — Regresión Logística

Baseline lineal. Con 7 clases muy solapadas esperamos que sea el peor modelo, pero sirve para confirmar que la no-linealidad de XGBoost y LightGBM aporta valor real.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        max_iter=2000,        # más iteraciones por el número de clases
        class_weight='balanced',
        random_state=11
    ))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("=== Regresión Logística ===")
print(classification_report(y_test, y_pred_lr, target_names=le_target_eleccion.classes_))


## Comparativa final — todos los modelos

In [ ]:
# reconstruyo base para comparar
modelo_base_ref = XGBClassifier(
    objective='multi:softmax', num_class=n_clases,
    n_estimators=100, max_depth=4, learning_rate=0.1,
    random_state=42, eval_metric='mlogloss'
)
pesos_base = compute_sample_weight(class_weight='balanced', y=y_train)
modelo_base_ref.fit(X_train, y_train, sample_weight=pesos_base)
y_pred_base = modelo_base_ref.predict(X_test)

modelos_todos = [
    ('Base XGB',         y_pred_base),
    ('XGB RndSearch',    y_pred_rxgb),
    ('LightGBM',         y_pred_lgbm),
    ('LGBM top15',       y_pred_top),
    ('LGBM drafteados',  y_pred_do),
    ('Random Forest',    y_pred_rf),
    ('SVM RBF',          y_pred_svm),
    ('Log. Regression',  y_pred_lr),
]

rows = []
for nombre, preds in modelos_todos:
    rep = classification_report(y_test, preds, target_names=le_target_eleccion.classes_, output_dict=True)
    rows.append({
        'modelo':   nombre,
        'macro_f1': round(rep['macro avg']['f1-score'], 4),
        'f1_ND':    round(rep['ND']['f1-score'], 4),
        'f1_1-10':  round(rep['1-10']['f1-score'], 4),
        'f1_51-60': round(rep['51-60']['f1-score'], 4),
    })

df_todos = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
print(df_todos.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0a1019')
ax.set_facecolor('#0a1019')

nombres = df_todos['modelo'].tolist()
x_pos = np.arange(len(nombres))
cols_m    = ['macro_f1', 'f1_1-10', 'f1_51-60']
colores_c = ['#f97316', '#A32D2D', '#854F0B']
etiq_c    = ['macro F1', 'F1 1-10', 'F1 51-60']
ancho_c   = 0.22

for i, (col, color, etiq) in enumerate(zip(cols_m, colores_c, etiq_c)):
    vals = df_todos[col].tolist()
    barras = ax.bar(x_pos + i * ancho_c, vals, ancho_c, label=etiq, color=color, alpha=0.85)
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                f'{b.get_height():.2f}', ha='center', va='bottom', fontsize=7, color='white')

ax.set_xticks(x_pos + ancho_c)
ax.set_xticklabels(nombres, color='white', fontsize=9, rotation=15, ha='right')
ax.set_ylim(0, 1.0)
ax.set_ylabel('valor', color='white')
ax.set_title('Comparativa todos los modelos — predicción de rango pick', color='white', fontsize=13, pad=12)
ax.tick_params(colors='white')
ax.legend(facecolor='#1a2535', labelcolor='white', fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor('#ffffff22')
plt.tight_layout()
plt.show()


In [ ]:
os.makedirs('../../pkl/modelos', exist_ok=True)
os.makedirs('../../pkl/preprocesado', exist_ok=True)

joblib.dump(mejor_xgb,        '../../pkl/modelos/xgb_rango_randsearch.pkl')
joblib.dump(lgbm_model,       '../../pkl/modelos/lgbm_rango_balanceado.pkl')
joblib.dump(lgbm_draft_only,  '../../pkl/modelos/lgbm_rango_solo_drafteados.pkl')
joblib.dump(lgbm_top,         '../../pkl/modelos/lgbm_rango_top15.pkl')
joblib.dump(rf_model,         '../../pkl/modelos/rf_rango.pkl')
joblib.dump(svm_pipeline,     '../../pkl/modelos/svm_rango.pkl')
joblib.dump(lr_pipeline,      '../../pkl/modelos/lr_rango.pkl')
joblib.dump(le_rango_draft,   '../../pkl/preprocesado/le_rango_draft_only.pkl')
joblib.dump(top15_vars,       '../../pkl/preprocesado/top15_vars_rango.pkl')

print("Modelos guardados")
